In [ ]:
# Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Dependencies and Repository
!pip install tdgl==0.8.4 cupy-cuda12x==13.6.0 papermill -qq
!git clone https://github.com/jczapata1/supercondtrans.git /content/SC -q

In [ ]:
import papermill as pm
import shutil
import json
import sys
import os

In [ ]:
# Work Directory
os.chdir('/content/SC/SuperCondTrans/Tdgl')
sys.path.insert(0, '/content/SC/SuperCondTrans/Tdgl')
sys.path.insert(0, '/content/SC/SuperCondTrans')

DRIVE_OUTPUT = '/content/drive/MyDrive/Other/TDGL_Output'

In [ ]:
# Apply Overrides
def apply_overrides(path, overrides):

    with open(path) as file:
        lines = file.readlines()

    result = []
    for line in lines:

        if ((':' in line) and (not line.strip().startswith('#'))):
            key = line.split(':')[0].strip()

            if (key in overrides):
                value     = overrides[key]
                value_str = f"'{value}'" if isinstance(value, str) else str(value)
                line      = f'{key} : {value_str}\n'

        result.append(line)

    with open(path, 'w') as file:
        file.writelines(result)

    return None

# Read Parameters
def read_param(path, key):

    with open(path) as file:

        for line in file:
            if ((':' in line) and (not line.strip().startswith('#'))):
                key, value = line.split(':', 1)
                if (key.strip() == key):
                    return value.strip().strip("'")

    return None

In [ ]:
# Json
PARAMS_JSON = '{"name":"run_001", "simulation":"Dynamic", "Input.in":{"gpu":true}}'

In [ ]:
# Parameters
PARAMS     = json.loads(PARAMS_JSON)
RUN_NAME   = PARAMS.get('name', 'run')
SIMULATION = PARAMS.get('simulation', 'Static')
FOLDER     = f'{DRIVE_OUTPUT}/{RUN_NAME}'

In [ ]:
# Input
input_overrides = PARAMS.get('Input.in', {})
apply_overrides('./input/Input.in', input_overrides)

# Profile and Disorder
profile  = read_param('./input/Input.in', 'profile')
disorder = read_param('./input/Input.in', 'disorder')

# Input Files
if ('Fields' in PARAMS): apply_overrides(f'./input/Fields/{profile}.in', PARAMS['Fields'])
if ('Epsilon' in PARAMS): apply_overrides(f'./input/Epsilon/{disorder}.in', PARAMS['Epsilon'])
if ('Current' in PARAMS): apply_overrides(f'./input/Current/{SIMULATION}.in', PARAMS['Current'])

In [ ]:
# Delete Default Files
for files in ['./input/Setup', './output/Static', './output/Dynamic', './output/Sweep']: shutil.rmtree(files); os.makedirs(files)

In [ ]:
# Run Setup
_ = pm.execute_notebook('./Setup.ipynb', '/tmp/Setup.ipynb', kernel_name='python3')
shutil.copytree('./input/Setup', f'{FOLDER}/Setup', dirs_exist_ok=True)

In [ ]:
# Run Simulation
txt = open('./Simulation.ipynb').read()
open('./Simulation.ipynb', 'w').write(txt.replace("simulation = 'Static'", f"simulation = '{SIMULATION}'"))

_ = pm.execute_notebook('./Simulation.ipynb', '/tmp/Simulation.ipynb', kernel_name='python3')
shutil.copytree(f'./output/{SIMULATION}', f'{FOLDER}/{SIMULATION}', dirs_exist_ok=True)

In [ ]:
# Zip
shutil.make_archive(f'{DRIVE_OUTPUT}/{RUN_NAME}', 'zip', DRIVE_OUTPUT, RUN_NAME)